# Lecture 30 - How Much? Integrating Data
### EES 3350/5350 · Python in Earth Science · Monday, November 16, 2026 

A derivative answers **"how fast?"**: how steeply the Cedar River drops, meter by meter. Today's tool answers the opposite question: **"how much, in total?"** In calculs, that's teh **integral**. With data, it's the **area under the curve**.

Back to the March 27-28, 2021 Nashville flood (Lecture 25). A gauge measured **discharge**: how many cubic feet of water flow past it **each second**. But as city planner, an insurer, or a reservoir operator asks a different question: **how much water came down the river, in total?** That's a **volume**, and it's the area under the hydrograph.

**Learning Objectives**:
- explain an integral of data as the **area under the curve**, and get its units (ft$^3$/s x s = ft$^3$)
- find a totaql with `integrate.trapezoid`, and a running total with `integrate.cumulative_trapezoid`
- see that a derivative and an integral **undo each other**
- turn a flood volume into a **runoff depth**: how deep a layer of water is there in the whole watershed

---
## Part 0 : set up

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import integrate

folder = "../../course-materials/week11/usgs/"

---
## Part 1 : volume is the area under the hydrograph
Let's start with a map:
![Nashville area with 3 gauges from the 2021 floods](../figures/Nash_2021_flood_map.png)
Start with the Harpeth River near Kingston Springs:

In [ ]:
harpeth = pd.read_csv(folder + "harpeth_daily.csv")
harpeth["time"] = pd.to_datetime(harpeth["time"])
harpeth = harpeth.sort_values("time", ignore_index=True)

t = np.array(harpeth["time"])
q = np.array(harpeth["value"])

print(len(q), "days")

### Figure 1

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.5))

ax.fill_between(t, q, color="tab:blue", alpha=0.3)
ax.plot(t, q, color="tab:blue")

ax.set_ylabel("daily mean discharge (ft$^3$/s)")
ax.set_title("Harpeth River near Kingston Springs, Feb-Apr 2021");

#### **Remember** `ax.fill_between(x, y)` shades the area between the line and zero.
That shaded area **is** the volume of water that passed the gauge. 

**Notice we didn't use a log scale this time:** on a log axis, areas don't mean anything, because the same height of shading can stand for wildly different amounts of water.

### The units of area

An area under a curve has the units of $y$ **times** the units of $x$ (a derivative was $y$ / $x$):
$$  \frac{\text{ft}^3}{s} \times s =  \text{ft}^3   $$

So the time axis has to be in **seconds**. There are 60 x 60 x 24 or **86,400 seconds in a day**.

### By hand: one day at a time
Each value is a **daily mean**, so the volume for one day is that day's discharge x 86,400 seconds: a tall, thin **rectangle**. 
![integrals visualized](../figures/Nash_2021_flood_integral_visualized.png)

#### To get the total, we add up all the rectangles:

In [ ]:
seconds_per_day = 60 * 60 * 24

one_day = q[0] * seconds_per_day
print(f"first day: {one_day:.0f} ft3")

total_rectangles = np.sum(q) * seconds_per_day
print(f"all {len(q)} days: {total_rectangles:,.0f} ft3")

<div class="alert alert-success">

**Tip:** REMEMBER `:,.0f` in an f-string means "no decimal places, with commas between the thousands." It makes big numbers readable: `12,345,678` instead of `12345678.0`.

</div>

---
## Part 2 : `integrate.trapezoid`
Rectangles work here because each value is an average over a whole day. But many records are **snapshots**: a reading every 15 minutes, a sample every week, an elevation every 10 m along a river. For snapshots, the usual way to find the area is to connect neighboring poitns with straight lines and add up the **trapezoids** underneath:

$$   \text{area of one trapezoid} = \frac{y_k + y_{k+1}}{2} \times \Delta x   $$

That's exactly what `integrate.trapezoid` does. You give it the $y$ values, and the spacing between points with `dx=` and it does it all for you:

In [ ]:
total_trapezoid = integrate.trapezoid(q, dx=seconds_per_day)

print(f"rectangles: {total_rectangles:,.0f} ft3")
print(f"trapezoids: {total_trapezoid:,.0f} ft3")

#### These answers are very close.
They differnt only at the **ends**: a trapezoid rule counts the first and last points as half-days, because each sits at the edge of only one trapezoid. With 89 days, those two half-days barely matter.

|**method**|**best for**|
|--|--|
|rectangles:`np.sum(y) * dx`| values that are **averages** over each interval (like daily means)|
|trapezoids:`integrate.trapezoid(y, dx=...)`| values that are **snapshots** at points (the usual case)|

<div class="alert alert-danger">

**Common Error**: Forgetting `dx=`. Without it, `trapezoid` assumes the points are **1 unit** apart. Here, that means 1 second instead of 1 day, so your volume comes out **86,400 times too small**, and nothing warns you. **Units of an integral = units of $y$ x units of $x$**. Check them every time.
    
</div>

### How big is that?
A cubic foot is hard to picture. Let's make it universal science by converting it to cubic meters (1 ft$^3$ = 0.0283 m$^3$), then compare with something familiar: an Olympic sized swimming pool (50 m x 25 m x 2 m deep) holds at least **2,500 m$^3$**.

In [ ]:
total_m3 = total_trapezoid * 0.0283

pools = total_m3 / 2500

print(f"{total_m3:,.0f} m3")
print(f"about {pools:,.0f} Olympic sized swimming pools")

<div class="alert alert-success">

**Tip**: If your records **aren't** evenly spaced, give `trapezoid` the actual $x$ values instead of `dx`: `integrate.trapezoid(y, x)`. It works just like `np.gradient(y, x)`.
    
</div>

---
## Part 3 : running a total with `cumulative_trapezoid`

`trapezoid` gives **one** number: the total. Often you want to watch the total **build up**: how much water passed by each day? That's a **running total**, like `.cumsum()` (from Lecture 24), but with trapezoids and the spacing built in.

#### We can do this using `integrate.cumulative_trapezoid`.

In [ ]:
running = integrate.cumulative_trapezoid(q, dx=seconds_per_day, initial=0)

print(len(q), len(running))
print(f"last value: {running[-1]:,.0f} ft3")

`integrate.cumulative_trapezoid` gives you back an array which stores the total volume **so far**, at every day. 
    
`initial=0` starts the running total at 0 on the first day, so the result is the **same length** as the data. (Without it, you'd get one fewer value, `.diff()`.) Its last value matches `trapezoid`'s total.

### Figure 2 : the rate, and the running total

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(9, 6), sharex=True)

axes[0].fill_between(t, q, color="tab:blue", alpha=0.3)
axes[0].plot(t, q, color="tab:blue")
axes[0].set_ylabel("discharge (ft$^3$)")
axes[0].set_title("Harpeth River near Kingston Springs, Feb-Apr 2021")

axes[1].plot(t, running / 1e9, color="black")
axes[1].set_ylabel("volume so far (billion ft$^3$)");


<div class="alert alert-warning">

### Question
Look at the bottom panel. Where does the running total climb most **steeply**? Where is it nearly **flat**? What do you see in the top panel at those same times?
    
</div>

> your answer here

<details>
<summary style="color:green;"><strong>Answer</strong></summary>

It climbs most steeply during the flood at the end of March, when the most water was passing **each second**, and it's nearly flat when the river is low. **The steepness of the running total is the discharge.** In other words, the discharge is the **derivative** of the running total, and the running total is the **integral** of the discharge.

</details>

### A derivative undoes an integral

**Just a brief check**: if we take Wednesday's `np.gradient` of the running total (with the same 86,400-second spacing) and compare it with the discharge we started with, we should end up with the same graph of discharge.

In [ ]:
back = np.gradient(running, seconds_per_day)

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(t, q, color="tab:blue", linewidth=4, alpha=0.4, label="measured discharge")
ax.plot(t, back, color="black", linewidth=1, label="np.gradient of the running total")
ax.set_ylabel("discharge (ft$^3$/s)")
ax.legend();

We can see that the lines sit almost on top of each other. So remember:
- **integrating** turns "ft$^3$ per second" into "ft$^3$ so far"
- **differentiating** turns it back

### That's just calculus!

|**operation**|**question it answers**|**units**|**tool**|
|:--|:--|:--|:--|
|derivative (Lecture 28)|how **fast** is it changing?|$y$ ÷ $x$|`np.gradient(y, x)`|
|integral (today)|how **much**, in total?|$y$ × $x$|`integrate.trapezoid(y, x)`|
|running integral (today)|how much **so far**?|$y$ × $x$|`integrate.cumulative_trapezoid(y, x, initial=0)`|

---
## Part 4 : just the flood

<div class="alert alert-info">

### Exercise 1 : how much of the season's water came in the flood?

Mask the Harpeth record to **March 25 through April 10** (thte flood and it's recession), and find the volume for just those days. What **percent** of the whole Feb-Apr volume passed in those 17 days?
    
</div>

In [ ]:
after_start = _______________ # make your mask part 1
before_end = _______________ # make your mask part 2
in_flood = _______________ # combine them

q_flood = np.array(harpeth["value"][____])

flood_volume = integrate.trapezoid(____, dx=____)
percent = flood_volume / total_trapezoid * 100

print(f"flood volume: d{flood_volume:,.0f} ft3")
print(f"{percent:.0f}% of Feb-Apr total")

<div class="alert alert-info">

### Exercise 2 : runoff depth for all three watersheds

A big river carries more water just because it drains more land. To compare fairly, divide each flood volume by the **drainage area**. Volume / area = a **depth**: how thick a layer of water, spread over the whole watershed, ran past the gauge. Hydrologists call it **runoff depth**, and it can be compared directly with rainfall.

The function below does it for one gauge. Complete it and then run the loop for all three gauges. The drainage area in the `_site.csv` files is in **square miles**, and 1 square mile = 27,878,400 ft$^2$.
    
</div>

In [ ]:
def runoff_depth(name):
    """Flood volume (Mar 25 - Apr 10, 2021) and runoff depth in inchees for one gauge."""
    daily = pd.read_csv(folder + name + "_daily.csv")
    daily["time"] = pd.to_datetime(daily["time"])
    daily = daily.sort_values("time", ignore_index=True)

    after_start = daily["time"] >= "2021-03-25"
    before_end = daily["time"] <= "2021-04-10"
    in_flood = after_start & before_end
    q_flood = np.array(daily["value"][in_flood])

    volume_ft3 = integrate.trapezoid(_____, dx=_____)

    site = pd.read_csv(folder + name + "_site.csv")
    area_sqmi = site["drainage_area"][0]
    area_ft2 = area_sqmi * _____

    depth_ft = volume_ft3 / _____
    depth_in = depth_ft * 12
    return volume_ft3, depth_in

for name in ["millcreek", "harpeth", "cumberland"]:
    volume, depth = runoff_depth(name)
    print(f"{name:12s} volume {volume:16,.0f} ft3   runoff depth {depth:5.1f} in")

<div class="alert alert-warning">

### Quesiton
The heaviest rain fell right over Nashville. Look up the two-day rainfall total for March 27-28, 2021 (the National Weather Service Nashivlle office has a summary page for the event). How does Mill Creek's runoff depth compare with the rainfall? Where could the rest of the rain have gone?
    
</div>

> your answer here

<details>
<summary style="color:green;"><strong>Answer</strong></summary>

The runoff depth should be **less** than the rainfall: not all the rain runs off. Some soaks into the soil and groundwater, some evaporates or is taken up by plants, and some is still draining after April 10. The **fraction** that runs off (runoff ÷ rainfall, the **runoff ratio**) is usually higher in urban watersheds like Mill Creek, where pavement and roofs keep water from soaking in.

</details>

---
## A useful table

|**you want**|**write**|
|:--|:--|
|import SciPy's integration tools|`from scipy import integrate`|
|total area under a curve (evenly spaced)|`integrate.trapezoid(y, dx=spacing)`|
|total area under a curve (any spacing)|`integrate.trapezoid(y, x)`|
|running total, same length as the data|`integrate.cumulative_trapezoid(y, dx=spacing, initial=0)`|
|total of interval **averages**|`np.sum(y) * spacing`|
|shade the area under a line|`ax.fill_between(x, y)`|
|shade between two lines, only where one is higher|`ax.fill_between(x, y1, y2, where=y2 > y1)`|
|big numbers with commas|`f"{value:,.0f}"`|

**Units of an integral = units of $y$ × units of $x$.** ft³/s × s = ft³. Volume ÷ area = depth.

